In [1]:
import langgraph as lg


In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, END

class MonEtat(TypedDict):
    counter: int
    prompt: str
    is_final: bool

def valider(etat: MonEtat):
    return {"est_valide": len(etat["prompt"].strip()) > 0}

graphe = StateGraph(MonEtat)
graphe.add_node("valider", valider)
graphe.set_entry_point("valider")
graphe.add_edge("valider", END)

app = graphe.compile()
resultat = app.invoke({"prompt": "Bonjour LangGraph", "est_valide": False})
print(resultat)

{'texte': 'Bonjour LangGraph', 'est_valide': True}


In [7]:
from typing import TypedDict
from langgraph.graph import StateGraph, END

class EtatBoucle(TypedDict):
    texte: str
    is_final: bool
    tour: int

def travailler(etat: EtatBoucle):
    tour = etat["tour"] + 1
    fini = ("ok" in etat["texte"].lower()) or (tour >= 3)
    return {"tour": tour, "is_final": fini}

def route(etat: EtatBoucle):
    if etat["is_final"]:
        return "fin"
    return "encore"

g = StateGraph(EtatBoucle)
g.add_node("travailler", travailler)
g.set_entry_point("travailler")
g.add_conditional_edges(
    "travailler",
    route,
    {
        "fin": END,
        "encore": "travailler",
    },
)

app_boucle = g.compile()
resultat_boucle = app_boucle.invoke({"texte": "pas encore", "is_final": False, "tour": 0})
print(resultat_boucle)

{'texte': 'pas encore', 'is_final': True, 'tour': 3}


In [ ]:
from typing import TypedDict, Optional
from langgraph.graph import StateGraph, END


class JavaAgentState(TypedDict):
    project_dir: str
    prompt: str
    java_code: str
    compile_ok: bool
    test_ok: bool
    last_error: str
    iteration: int
    max_iterations: int
    is_final: bool
    patch_target: Optional[str]


# ---------- MCP placeholders (a remplacer par tes vrais calls MCP) ----------
def mcp_plan_java_task(prompt: str) -> str:
    # Exemple: appeler un tool MCP de planification
    return f"Plan for: {prompt}"


def mcp_generate_java_code(prompt: str, last_error: str = "") -> str:
    # Exemple: appeler un tool MCP/LLM pour generer du Java
    return (
        "public class Calculator { public static int add(int a,int b){ return a+b; } }"
    )


def mcp_write_java_file(project_dir: str, java_code: str) -> bool:
    # Exemple: tool MCP write_file
    return True


def mcp_compile_and_test(project_dir: str) -> tuple[bool, bool, str]:
    # Exemple: tool MCP validate_code / mvn test
    # Retour: (compile_ok, test_ok, last_error)
    return (False, False, "Placeholder compile error: method multiply missing")


def mcp_patch_java_code(java_code: str, error: str) -> str:
    # Exemple: tool MCP de patch base sur les erreurs
    patched = java_code
    if "multiply" in error and "multiply" not in patched:
        patched = patched.replace(
            "}", " public static int multiply(int a,int b){ return a*b; } }"
        )
    return patched


# ---------- Nodes ----------
def plan_node(state: JavaAgentState):
    plan = mcp_plan_java_task(state["prompt"])
    return {"patch_target": plan}


def generate_node(state: JavaAgentState):
    code = mcp_generate_java_code(state["prompt"], state["last_error"])
    return {"java_code": code}


def write_node(state: JavaAgentState):
    _ = mcp_write_java_file(state["project_dir"], state["java_code"])
    return {}


def validate_node(state: JavaAgentState):
    compile_ok, test_ok, err = mcp_compile_and_test(state["project_dir"])
    done = compile_ok and test_ok
    return {
        "compile_ok": compile_ok,
        "test_ok": test_ok,
        "last_error": err,
        "is_final": done,
        "iteration": state["iteration"] + 1,
    }


def patch_node(state: JavaAgentState):
    patched = mcp_patch_java_code(state["java_code"], state["last_error"])
    return {"java_code": patched}


def route_after_validate(state: JavaAgentState):
    if state["is_final"]:
        return "done"
    if state["iteration"] >= state["max_iterations"]:
        return "done"
    return "patch"


# ---------- Graph ----------
g_java = StateGraph(JavaAgentState)
g_java.add_node("plan", plan_node)
g_java.add_node("generate", generate_node)
g_java.add_node("write", write_node)
g_java.add_node("validate", validate_node)
g_java.add_node("patch", patch_node)

g_java.set_entry_point("plan")
g_java.add_edge("plan", "generate")
g_java.add_edge("generate", "write")
g_java.add_edge("write", "validate")
g_java.add_conditional_edges(
    "validate",
    route_after_validate,
    {
        "done": END,
        "patch": "patch",
    },
)
g_java.add_edge("patch", "write")

app_java = g_java.compile()

# ---------- Run demo ----------
initial_state: JavaAgentState = {
    "project_dir": "./generated_java_project",
    "prompt": "Genere une classe Calculator avec add et multiply",
    "java_code": "",
    "compile_ok": False,
    "test_ok": False,
    "last_error": "",
    "iteration": 0,
    "max_iterations": 3,
    "is_final": False,
    "patch_target": None,
}

result_java = app_java.invoke(initial_state)
print(result_java)

{'project_dir': './generated_java_project', 'prompt': 'Genere une classe Calculator avec add et multiply', 'java_code': 'public class Calculator { public static int add(int a,int b){ return a+b;  public static int multiply(int a,int b){ return a*b; } }  public static int multiply(int a,int b){ return a*b; } }', 'compile_ok': False, 'test_ok': False, 'last_error': 'Placeholder compile error: method multiply missing', 'iteration': 3, 'max_iterations': 3, 'is_final': False, 'patch_target': 'Plan for: Genere une classe Calculator avec add et multiply'}
